In [ ]:
# phrase question 3
sentence_q3 = "Je suis tombé du toit de l’usine"
# phrase question 6
sentence_q6 = "EN: I fell from the roof"

# TP2 - Estimateur de Similarités

### Question 1: 

En Python, un nombre flottant est représenté par 8 octets. Pour un vecteur de 300 dimensions, la taille totale en octets est :

300 dimensions * 8 octets par dimension = 2400 octets

Donc, la représentation vectorielle en 300 dimensions d’un seul mot nécessite en moyenne 2400 octets de stockage.

### Question 2:

In [ ]:
import numpy as np

In [ ]:
max_memory = 100 * np.power(2, 20)  # 100 Mo en octets
vector_size = 2400  # 300 dimensions * 8 octets par dimension
max_words_in_memory = max_memory // vector_size # Nombre maximal de mots qu'on peut charger en mémoire

print(f"Nombre maximal de mots en mémoire : {max_words_in_memory}")

In [ ]:
# Fonction pour obtenir le vecteur d'un mot
def get_vector(word, filepath, max_words_in_memory):
    with open(filepath, 'r', encoding='utf-8') as file:
        lines = []
        for line in file:
            lines.append(line.strip())
            if len(lines) >= max_words_in_memory:
                result = process_lines(lines, word)
                if result is not None:
                    return result
                lines = []
        if lines:
            result = process_lines(lines, word)
            if result is not None:
                return result
    return None

# Fonction pour traiter les lignes et trouver le vecteur du mot
def process_lines(lines, word):
    for line in lines:
        parts = line.split()
        if parts[0] == word:
            return np.array([float(x) for x in parts[1:]])
    return None


filepath = '/media/tessier/FORMATE/cc.fr.300.vec'
word = 'exemple'
# Obtentions des vecteurs de word
vector = get_vector(word, filepath, max_words_in_memory)
print(vector)

### Question 3 :

In [ ]:
# Fonction pour diviser une phrase en mots et/ou signes de ponctuation
def split_sentence(sentence):
    tokens = []
    word = ''
    for char in sentence:
        if char.isalnum() or char == "'":
            word += char
        else:
            if word:
                tokens.append(word)
                word = ''
            if char.strip():
                tokens.append(char)
    if word:
        tokens.append(word)
    return tokens

# Fonction pour obtenir le vecteur moyen d'une phrase
def get_average_vector(sentence, filepath, max_words_in_memory):
    tokens = split_sentence(sentence)
    vectors = []
    for token in tokens:
        vector = get_vector(token, filepath, max_words_in_memory)
        if vector is not None:
            vectors.append(vector)
        else:
            print(f"Vector for '{token}' not found")
    
    if vectors:
        average_vector = np.mean(vectors, axis=0)
        return average_vector
    else:
        return None

# Calcul de la moyenne des vecteurs pour la phrase
average_vector = get_average_vector(sentence_q3, filepath, max_words_in_memory)
if average_vector is not None:
    print(f"Average vector: {average_vector}")
else:
    print("No vectors found for the given sentence")

### Question 4 :

In [ ]:
# Fonction pour calculer la similarité cosinus entre deux vecteurs
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

word1 = 'exemple'
word2 = 'test'

# Calcul du vecteur moyen de word1
vector1 = get_vector(word1, filepath, max_words_in_memory)
# Calcul du vecteur moyen de word2
vector2 = get_vector(word2, filepath, max_words_in_memory)

if vector1 is not None and vector2 is not None:
    # Calcul de la similarité entre word1 et word2
    similarity = cosine_similarity(vector1, vector2)
    print(f"Similarité de cosinus entre '{word1}' et '{word2}' : {similarity}")
else:
    print("L'un des mots ou les deux n'ont pas été trouvés dans le fichier de vecteurs")

### Question 5 :

In [ ]:
# Fonction pour trouver le mot le plus proche du vecteur moyen de la phrase
def find_closest_word(average_vector, filepath, max_lines):
    if average_vector is None:
        return None

    closest_word = None
    closest_distance = -1

    with open(filepath, 'r', encoding='utf-8') as file:
        for line_number, line in enumerate(file):
            if line_number >= max_lines:
                break
            parts = line.strip().split(' ')
            word = parts[0]
            vector = np.array([float(value) for value in parts[1:]])
            if vector.shape != average_vector.shape:
                continue  
            distance = cosine_similarity(average_vector, vector)
            if closest_word is None or distance > closest_distance:
                closest_word = word
                closest_distance = distance

    return closest_word, closest_distance

if average_vector is not None:
    # Trouve le mot le plus proche du vecteur moyen calculé
    closest_word, closest_similarity = find_closest_word(average_vector, filepath, max_words_in_memory)
    print("Le mot le plus proche est",closest_word, "avec une similarité de", closest_similarity)   
else:
    print("L'analyse de la phrase n'a pas produit de vecteur moyen")

### Question 6:

In [ ]:
# Fonction pour extraire le préfixe de la phrase
def extract_prefix_and_sentence(input_text):
    parts = input_text.split(': ', 1)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()
    else:
        return 'FR', input_text.strip()

# Extraction du préfixe et de la phrase
lang_prefix, sentence = extract_prefix_and_sentence(sentence_q6)

# Définis le chemin du fichier de vecteurs en fonction du préfixe de langue
if lang_prefix == 'FR':
    filepath = 'data/vector-fr.txt'
elif lang_prefix == 'EN':
    filepath = 'data/vector-en.txt'
else:
    raise ValueError("Unsupported language prefix")

# Calculer le vecteur moyen de la phrase
average_vector = get_average_vector(sentence, filepath, max_words_in_memory)

if average_vector is not None:
    # Trouver le mot le plus proche du vecteur moyen calculé
    closest_word, closest_similarity = find_closest_word(average_vector, filepath, max_words_in_memory)
    print("Le mot le plus proche est",closest_word, "avec une similarité de", closest_similarity)   
else:
    print("L'analyse de la phrase n'a pas produit de vecteur moyen")